# 2.3 - Membangun AI Agent Tools Menggunakan Unity Catalog (UC) Functions

Notebook ini mempelajari cara membuat dan mendaftarkan **Tools** untuk agen AI (*Tool-Calling Agent*) menggunakan **Unity Catalog (UC) Functions**. 

Di ekosistem Databricks, agen cerdas (LLM) dapat menggunakan fungsi SQL maupun Python yang terdaftar di Unity Catalog sebagai alat (*tools*) untuk mengeksekusi logika bisnis, query database secara real-time, dan melakukan aksi transaksional secara aman di bawah tata kelola (*governance*) terpusat.

### Ringkasan Proses:
1. **Instalasi Pustaka Pendukung**: Menginstal `unitycatalog-ai` untuk integrasi fungsi Unity Catalog dengan framework agen AI.
2. **Setup Lingkungan Lab**: Menginisialisasi katalog, skema, tabel sampel produk, dan data inventaris melalui `Lab_Setup_02`.
3. **Eksplorasi Data**: Memeriksa tabel katalog smartphone (`smartphones`) dan data stok barang (`inventory_stock`).
4. **Pembuatan UC Tools Berbasis SQL**:
   - `get_latest_phone()`: Fungsi SQL tanpa parameter untuk mengambil produk smartphone terbaru.
   - `check_inventory(phone_id)`: Fungsi SQL terparameterisasi untuk mengecek kuantitas sisa stok barang.
   - `get_current_user()`: Fungsi SQL untuk mengambil identitas email/username pengguna sesi yang sedang aktif.
5. **Pembuatan & Pendaftaran UC Tools Berbasis Python**:
   - Menulis fungsi Python `generate_checkout_url` untuk membuat tautan pembayaran pesanan dengan validitas 30 menit.
   - Menguji fungsi Python secara lokal.
   - Mendaftarkan fungsi Python tersebut langsung ke Unity Catalog menggunakan SDK `unitycatalog.ai`.


### Langkah 1: Instalasi Library Pendukung
Menginstal pustaka `unitycatalog-ai` untuk mengintegrasikan fungsi Unity Catalog sebagai *tools* yang dapat dipanggil oleh LLM, kemudian merestart sesi Python agar perubahan library langsung aktif.


In [0]:
# Menginstal pustaka unitycatalog-ai untuk mengonversi fungsi Unity Catalog menjadi tools agen AI
%pip install --quiet --upgrade unitycatalog-ai
# Merestart kernel Python agar pustaka yang baru diinstal dapat langsung digunakan
%restart_python

### Langkah 2: Setup Lingkungan Lab
Menjalankan notebook setup lab untuk menginisialisasi katalog, skema, dan tabel-tabel data pendukung (`smartphones`, `inventory_stock`).


In [0]:
%run ../Includes/Lab_Setup_02

### Langkah 3: Eksplorasi Data Katalog Produk
Melihat isi tabel `smartphones` yang mencatat informasi produk smartphone (seperti model, warna, harga, dan tanggal rilis).


In [0]:
%sql
-- Menampilkan seluruh daftar produk smartphone pada tabel 'smartphones'
SELECT * FROM smartphones

### Langkah 4: Eksplorasi Data Inventaris Stok
Melihat isi tabel `inventory_stock` yang mencatat jumlah ketersediaan stok fisik untuk setiap ID produk smartphone.


In [0]:
%sql
-- Menampilkan data kuantitas stok produk yang tersedia pada tabel 'inventory_stock'
SELECT * FROM inventory_stock

### Langkah 5: Membuat UC SQL Function - `get_latest_phone`
Mendefinisikan fungsi SQL terkelola di Unity Catalog untuk mengambil model smartphone terbaru.
> **Penting untuk AI Agent**: Klausul `COMMENT` pada fungsi bertindak sebagai *deskripsi tool* yang dibaca dan dipahami oleh LLM ketika model memutuskan kapan dan bagaimana memanggil fungsi tersebut.


In [0]:
%sql
-- Membuat atau memperbarui fungsi SQL terkelola di Unity Catalog
-- Fungsi ini bertindak sebagai Tool untuk AI Agent dalam mencari produk smartphone terbaru
CREATE OR REPLACE FUNCTION get_latest_phone()
RETURNS TABLE(phone_id INT, model STRING, color STRING, price INT)
-- COMMENT sangat penting karena teks ini menjadi petunjuk (tool description) bagi LLM
COMMENT 'Returns the latest phone, ordered by release_date descending. IMPORTANT: This function accepts ZERO parameters. Do not pass any arguments.'
RETURN
  SELECT phone_id, model, color, price
  FROM smartphones
  ORDER BY release_date DESC
  LIMIT 1;

### Langkah 6: Pengujian Fungsi `get_latest_phone`
Menjalankan kueri SQL untuk memvalidasi output dari fungsi `get_latest_phone()`.


In [0]:
%sql
-- Menguji pemanggilan fungsi get_latest_phone() untuk memastikan output sesuai ekspektasi
SELECT * FROM get_latest_phone()

### Langkah 7: Membuat UC SQL Function - `check_inventory`
Mendefinisikan fungsi SQL terparameterisasi untuk mengecek stok berdasarkan `phone_id`.
Komentar pada parameter (`COMMENT 'ID of the phone to check...'`) akan dipetakan menjadi deskripsi argumen pada skema JSON Tool Calling LLM.


In [0]:
%sql
-- Membuat fungsi SQL terparameterisasi untuk memeriksa ketersediaan stok produk
CREATE OR REPLACE FUNCTION check_inventory(phone_id INT COMMENT 'ID of the phone to check (integer)')
RETURNS TABLE (
  item_id INT,
  quantity INT
)
-- Deskripsi fungsi yang akan dibaca LLM saat memutuskan untuk memanggil tool ini
COMMENT 'Checks the stock quantity for a given phone in the inventory.'
RETURN
  SELECT item_id, quantity
  FROM inventory_stock
  WHERE item_id = phone_id;

### Langkah 8: Pengujian Fungsi `check_inventory`
Menguji eksekusi fungsi `check_inventory` menggunakan sampel ID telepon (`phone_id = 4`).


In [0]:
%sql
-- Menguji pemanggilan fungsi check_inventory dengan sampel parameter phone_id = 4
SELECT * FROM check_inventory(4)

### Langkah 9: Membuat UC SQL Function - `get_current_user`
Mendefinisikan fungsi SQL tanpa parameter untuk mengambil informasi sesi user aktif (`current_user()`). Tool ini berguna bagi agen untuk mengetahui siapa pengguna yang sedang berinteraksi.


In [0]:
%sql
-- Membuat fungsi SQL untuk mengambil email/username pengguna yang sedang aktif dalam sesi
CREATE OR REPLACE FUNCTION get_current_user()
RETURNS STRING
-- Komentar petunjuk bagi LLM bahwa fungsi ini tidak memerlukan parameter apa pun
COMMENT 'Returns the current session user, used for identifying the active user. IMPORTANT: This function accepts ZERO parameters. Do not pass any arguments.'
RETURN current_user();

### Langkah 10: Pengujian Fungsi `get_current_user`
Menguji eksekusi fungsi `get_current_user()` untuk memastikan identitas pengguna sesi berhasil diperoleh.


In [0]:
%sql
-- Menguji pemanggilan fungsi get_current_user() untuk memeriksa identitas sesi aktif
SELECT get_current_user()

### Langkah 11: Mendefinisikan Fungsi Bisnis Python - `generate_checkout_url`
Menulis fungsi Python untuk membuat URL transaksi checkout dengan waktu kedaluwarsa 30 menit.
Docstring dan *type hinting* Python yang lengkap sangat penting karena SDK `unitycatalog-ai` akan mengonversinya secara otomatis menjadi skema deskripsi tool untuk LLM.


In [0]:
def generate_checkout_url(user_id: str, phone_id: int, valid_quantity: int) -> str:
    """
    Returns the checkout URL to fulfill the user's order for a phone with a given quantity.
    The URL is valid for 30 minutes.

    Args:
        user_id (str): Identifier of the current user/session.
        phone_id (int): ID of the phone to buy.
        valid_quantity (int): the requested quantity, validated against current stock

    Returns:
        str: Checkout URL string.
    """
    from datetime import datetime, timedelta
    from urllib.parse import urlencode

    # Menghitung batas waktu kedaluwarsa tautan checkout (berlaku 30 menit ke depan)
    expiry_time = datetime.now() + timedelta(minutes=30)
    
    # Menyusun parameter query string URL pesanan
    params = {
        "i": phone_id,
        "q": valid_quantity,
        "u": user_id,
        "t": expiry_time
    }
    
    # Menghasilkan URL checkout lengkap dengan parameter ter-encode
    checkout_url = f"https://www.derar.cloud/checkout?{urlencode(params)}"
    return checkout_url

### Langkah 12: Pengujian Fungsi Python secara Lokal
Menguji fungsi `generate_checkout_url` secara lokal di notebook untuk memverifikasi struktur URL yang dihasilkan.


In [0]:
# Menguji eksekusi fungsi generate_checkout_url secara lokal dengan data sampel
generate_checkout_url("derar@derar.cloud", 4, 2)

### Langkah 13: Mendaftarkan Fungsi Python ke Unity Catalog sebagai UC Tool
Menggunakan `DatabricksFunctionClient` dari paket `unitycatalog-ai` untuk mempublikasikan fungsi Python lokal langsung ke dalam Unity Catalog. Setelah terdaftar, fungsi ini dapat dipanggil lewat SQL dan dijadikan *tool* oleh agen AI.


In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

# Menginisialisasi client Databricks Function dari pustaka unitycatalog-ai
client = DatabricksFunctionClient()

# Mendaftarkan fungsi Python ke dalam Unity Catalog sebagai tool resmi untuk AI Agent
python_tool_uc_info = client.create_python_function(func=generate_checkout_url,
                                                    catalog=course.catalog,
                                                    schema=course.schema,
                                                    replace=True)